In [1]:
%%capture
!pip install facenet-pytorch

In [11]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

## Libraries

In [23]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
from torchsummary import summary

from PIL import Image

import numpy as np
np.bool = np.bool_

import mxnet as mx
from mxnet import recordio

from image_iter import FaceDataset

from tqdm import tqdm

## Functions

## Models

In [13]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [30]:
# Load model
'''
The cropped faces are passed as input in the CNN and we get an embedding for each face.
Important to set the model at .eval()
'''
teacher = InceptionResnetV1(
    classify=True,
    pretrained='casia-webface').to(device)

In [31]:
student = models.mobilenet_v3_small(pretrained=True)

## Load Data

In [32]:
BATCH_SIZE=32
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'

In [33]:
dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


## Training

In [37]:
len(train_loader)

15331

In [34]:
x, y = next(iter(train_loader))

In [35]:
ys = teacher(x.to(device, dtype=torch.float32))

In [36]:
ys.shape

torch.Size([32, 10575])

In [22]:
y.shape

torch.Size([32])

In [ ]:
for batch, X, labels in tqdm(enumerate(train_loader))